### ZCU216 Qubit Readout - PYNQ Driver 

Clean two-phase flow: a one-time **SETUP PHASE** that records the experiment's
base noise, then the **EXPERIMENT** with hardware noise-subtraction live.

- **SETUP PHASE - Dark Mode ON, one-time** (Step 12) - capture the static
  background (dark current, fixed-pattern read noise, stray light) into the
  sticky `base_noise` BRAM banks inside `roi_storage`. Persists until
  overwritten or the bitstream is reloaded.
- **EXPERIMENT - Dark Mode OFF** (Step 13) - `read_streamer` subtracts that
  baseline from every ROI, per pixel, `max(0, roi - noise)`, raising the
  signal-to-noise ratio of the Rydberg readout.
- **64-bit VDMA** - 8 px/beat, 64 beats/line; `HSIZE` stays 512 bytes.
- **`latency_capture` readout** - 15 FPGA registers (`s_axi_lat` @ 0xB0020000)
  read after every frame: cycle-accurate PL pipeline breakdown at 520 MHz.
- **Lean hot path** - frames are pre-loaded to RAM (Step 10c); `run_frame()`
  does no disk I/O and no sleeps. Remaining jitter is CPU/OS only - isolated
  and quantified by the determinism analysis (Step 14).

**Deterministic computing model:**
```
[cache flush] [DMA arm] [VDMA_VSIZE write] ------------------------ [DMA IOC]
                                  |   FPGA PL 520 MHz (deterministic)   |
                                  +- delta_pipe cycles (latency_capture)+
                                                  v
                               [vdma_stop] [cache inv] [decode]
```
The FPGA PL part is hardware-deterministic to <1 cycle. All jitter lives in CPU layers.


#### Step 1 - Frame generation (run once)


In [ ]:
"""
Frame generator - neutral-atom fluorescence model.

"""
import numpy as np, os
from PIL import Image

W, H           = 512, 512
NUM_QUBITS     = 100
GRID_COLS      = 10
QUBIT_START_X  = 26
QUBIT_START_Y  = 27
SIGMA_PSF      = 1.5
SIGNAL_PHOTONS = 220.0
BG_PHOTONS     = 1.8
READ_NOISE     = 1.0
CLIP_PHOTONS   = 250.0

def calc_qx(q, x_offset=0):
    x = QUBIT_START_X + x_offset
    for c in range(q % GRID_COLS): x += 52 if c == 4 else 51
    return x

def calc_qy(q, y_offset=0):
    y = QUBIT_START_Y + y_offset
    for r in range(q // GRID_COLS): y += 52 if r == 4 else 51
    return y

coords = [(calc_qx(q), calc_qy(q)) for q in range(NUM_QUBITS)]

def make_frame(excited, seed=42):
    rng = np.random.default_rng(seed)
    img = rng.poisson(BG_PHOTONS, (H, W)).astype(np.float32)
    img += rng.normal(0, READ_NOISE, (H, W)).astype(np.float32)
    r = int(np.ceil(4 * SIGMA_PSF))
    for q, (qx, qy) in enumerate(coords):
        if not excited[q]: continue
        x0,x1 = max(0,qx-r), min(W,qx+r+1)
        y0,y1 = max(0,qy-r), min(H,qy+r+1)
        xx,yy = np.meshgrid(np.arange(x0,x1), np.arange(y0,y1))
        psf   = SIGNAL_PHOTONS * np.exp(-((xx-qx)**2+(yy-qy)**2)/(2*SIGMA_PSF**2))
        img[y0:y1,x0:x1] += rng.poisson(psf.clip(0)).astype(np.float32)
    return img.clip(0)

def frame_to_uint8(f):
    return (f.clip(0, CLIP_PHOTONS) / CLIP_PHOTONS * 255).astype(np.uint8)

patterns = {
    "all_ground"  : [False] * NUM_QUBITS,
    "col_stripes" : [q % GRID_COLS < 5                for q in range(NUM_QUBITS)],
    "checkerboard": [(q//GRID_COLS+q%GRID_COLS)%2==0  for q in range(NUM_QUBITS)],
    "all_rydberg" : [True]  * NUM_QUBITS,
    "single_q0"   : [q == 0                            for q in range(NUM_QUBITS)],
    "row_stripes" : [q//GRID_COLS % 2 == 0             for q in range(NUM_QUBITS)],
}
os.makedirs('frames', exist_ok=True)
for idx, (name, excited) in enumerate(patterns.items()):
    arr = frame_to_uint8(make_frame(excited, seed=idx))
    Image.fromarray(arr, mode='L').save(f'frames/{name}.png')
    arr.tofile(f'frames/{name}.bin')
    print(f'  {name:<16} {sum(excited):>3}/100 Rydberg')
print('Frames written to frames/*.bin')


#### Step 1b - Dark calibration + verification frames
One-time calibration frames for the dark-mode / noise-subtraction path and the
runtime Gaussian-threshold check. These are **not** in the steady-state hot
loop - dark capture runs once to load the `base_noise` banks, then the
latency-critical `run_frame()` path is unchanged.

In [ ]:
"""
Dark-calibration + verification frames for the dark-mode / noise-subtraction
and runtime-threshold checks. 

  dark_zero.bin     all-zero closed-shutter frame. Captured into the
                    base_noise banks it zeroes the baseline -> per-pixel
                    saturating subtraction becomes a no-op. Also used to
                    RESET the baseline between tests.
  dark_pedestal.bin realistic noisy dark frame: a uniform DC pedestal
                    (dark current / read offset) plus Poisson shot noise,
                    no qubit blobs. Captured as the noise baseline it makes
                    read_streamer subtract a real, non-zero background.
  verify_active.bin deterministic near-threshold active frame:
                    flat background = DARK_PEDESTAL_DN, and a uniform glow
                    block on a known (checkerboard) excited set so that
                      excited 3x3 ROI = VERIFY_HI         -> score 16*VERIFY_HI
                      ground  3x3 ROI = DARK_PEDESTAL_DN  -> score 16*PED
                    Chosen so the decision is crisp and flips predictably
                    when (a) a pedestal baseline is subtracted, or
                    (b) the Gaussian threshold is moved.
"""
import numpy as np, os
from PIL import Image

# ---- pulled-out verification parameters --------------------------------
DARK_PEDESTAL_DN = 16    # closed-shutter DC pedestal, 8-bit DN
VERIFY_HI        = 32    # excited-qubit ROI level in verify_active
                         #   excited score = 16 * 32 = 512  (kernel weight sum = 16)
                         #   ground  score = 16 * 16 = 256
VERIFY_PATTERN   = [(q // GRID_COLS + q % GRID_COLS) % 2 == 0
                    for q in range(NUM_QUBITS)]   # checkerboard, 50/100

os.makedirs('frames', exist_ok=True)

# dark_zero -- all-zero closed shutter
dark_zero = np.zeros((H, W), dtype=np.uint8)
dark_zero.tofile('frames/dark_zero.bin')
Image.fromarray(dark_zero, mode='L').save('frames/dark_zero.png')

# dark_pedestal -- uniform DC pedestal + Poisson shot noise, no qubit blobs
_rng = np.random.default_rng(2024)
dark_ped = _rng.poisson(DARK_PEDESTAL_DN, (H, W)).astype(np.int32)
dark_ped = np.clip(dark_ped, 0, 255).astype(np.uint8)
dark_ped.tofile('frames/dark_pedestal.bin')
Image.fromarray(dark_ped, mode='L').save('frames/dark_pedestal.png')

# verify_active -- flat background + uniform glow blocks on excited qubits.
# A 5x5 uniform block centred on the qubit fully covers the 3x3 ROI that the
# RTL captures (rows calc_qy-1..+1, cols calc_qx-1..+1).
verify = np.full((H, W), DARK_PEDESTAL_DN, dtype=np.uint8)
for q, (qx, qy) in enumerate(coords):
    if not VERIFY_PATTERN[q]:
        continue
    y0, y1 = max(0, qy - 2), min(H, qy + 3)
    x0, x1 = max(0, qx - 2), min(W, qx + 3)
    verify[y0:y1, x0:x1] = VERIFY_HI
verify.tofile('frames/verify_active.bin')
Image.fromarray(verify, mode='L').save('frames/verify_active.png')

n_exc = sum(VERIFY_PATTERN)
print('Calibration / verification frames written to frames/:')
print(f'  dark_zero       all-zero            (baseline reset / no-op subtraction)')
print(f'  dark_pedestal   pedestal={DARK_PEDESTAL_DN} + Poisson  (realistic noisy dark frame)')
print(f'  verify_active   bg={DARK_PEDESTAL_DN}  glow={VERIFY_HI}  excited={n_exc}/100')
print(f'                  scores: excited=16*{VERIFY_HI}={16*VERIFY_HI}, '
      f'ground=16*{DARK_PEDESTAL_DN}={16*DARK_PEDESTAL_DN}')

#### Step 2 - FPGA programming


In [ ]:
import numpy as np, time
import matplotlib.pyplot as plt, matplotlib.patches as patches
from matplotlib.colors import PowerNorm
from collections import namedtuple
import pynq
from pynq import Overlay, allocate, MMIO

HWH_PATH = "/home/xilinx/jupyter_notebooks/Latency_Test_QICK_Diff/design_qubit_readout_wrapper.hwh"
BIT_PATH = "/home/xilinx/jupyter_notebooks/Latency_Test_QICK_Diff/design_qubit_readout_wrapper.bit"

print(f'PYNQ version : {pynq.__version__}')
print('Programming FPGA ...', end=' ', flush=True)
ol = Overlay(BIT_PATH)
print('done.')
print('Waiting for MMCM lock + proc_sys_reset ...', end=' ', flush=True)
time.sleep(2.0)
print('done.\n')
for name, d in ol.ip_dict.items():
    if 'phys_addr' in d:
        print(f"  {name:<40}  0x{d['phys_addr']:012X}")


#### Step 3 - IP handles and register constants


In [ ]:
# Discover IP addresses robustly 
# PYNQ ip_dict key = instance name from the HWH 'fullpath' attribute.
# This is usually the leaf name ('datastream_processor_0') BUT can include
# the BD wrapper prefix ('design_qubit_readout_i/datastream_processor_0')
# in some PYNQ/Vivado versions. Never hardcode the key; discover at runtime.

print('All IPs visible in ol.ip_dict:')
for k, v in ol.ip_dict.items():
    if 'phys_addr' in v:
        print(f"  '{k}'  0x{v['phys_addr']:010X}")

def _find_ip(d, *fragments):
    """Return (key, phys_addr) for the first ip_dict entry whose key contains all fragments."""
    for key, val in d.items():
        if 'phys_addr' not in val: continue
        if all(f.lower() in key.lower() for f in fragments):
            return key, val['phys_addr']
    raise KeyError(
        f'No IP matching {fragments} in ip_dict.\n'
        f"Keys: {[k for k in d if 'phys_addr' in d[k]]}")

_d = ol.ip_dict
_, COORD_BASE = _find_ip(_d, 'datastream_processor')
_, VDMA_BASE  = _find_ip(_d, 'vdma')
_, DMA_BASE   = _find_ip(_d, 'axi_dma')
LAT_BASE = COORD_BASE + 0x00020000   # s_axi_lat at COORD_BASE + 128 KB

for label, base in [('coord_ip', COORD_BASE), ('vdma', VDMA_BASE),
                     ('dma', DMA_BASE), ('lat_ip', LAT_BASE)]:
    print(f'  {label:<10}  0x{base:010X}')

# -- Thin MMIO wrapper ------------------------------------------------------
class _RawIP:
    def __init__(self, base, size):
        self._m = MMIO(base, size)
    def read(self, off):       return self._m.read(off)
    def write(self, off, val): self._m.write(off, val)

coord_ip = _RawIP(COORD_BASE, 0x0400)
vdma     = _RawIP(VDMA_BASE,  0x10000)
dma      = _RawIP(DMA_BASE,   0x1000)
lat_ip   = _RawIP(LAT_BASE,   0x0040)

VDMA_CR     = 0x00;  VDMA_SR     = 0x04
VDMA_VSIZE  = 0x50;  VDMA_HSIZE  = 0x54
VDMA_STRIDE = 0x58;  VDMA_ADDR1  = 0x5C
VDMA_ADDR2  = 0x60;  VDMA_ADDR3  = 0x64
S2MM_CR     = 0x30;  S2MM_SR     = 0x34
S2MM_DA     = 0x48;  S2MM_LEN    = 0x58

LAT_COUNTER    = 0x00;  LAT_FRAME_SEQ  = 0x04
LAT_TS_SOF     = 0x08;  LAT_TS_MATCH   = 0x0C
LAT_TS_ROI_WR  = 0x10;  LAT_TS_READY   = 0x14
LAT_TS_FIRST_R = 0x18;  LAT_TS_LAST_R  = 0x1C
LAT_TS_FDONE   = 0x20;  LAT_D_PIPE     = 0x24
LAT_D_MATCH    = 0x28;  LAT_D_ROI      = 0x2C
LAT_D_STORAGE  = 0x30;  LAT_D_READOUT  = 0x34
LAT_D_FILTER   = 0x38

# ---- qubit_lookup_axi control registers (shared with the coord AXI slave) ----
# Address map (qubit_lookup_axi.sv):
#   0x000 .. 0x31F   qubit X/Y coordinate pairs (100 * 8 bytes)
#   0x320            reg_ctrl          -- bit 0 = dark_mode
#   0x324            reg_gauss_thresh  -- [15:0] = Gaussian decision threshold
CTRL_DARK_MODE          = 0x320
CTRL_GAUSS_THRESH       = 0x324
GAUSS_THRESHOLD_DEFAULT = 500   # mirrors params_pkg::GAUSS_THRESHOLD_DEFAULT
DARK_MODE_ACTIVE        = 0     # normal readout : ping-pong banks + frame_ready
DARK_MODE_CAPTURE       = 1     # dark capture   : ROIs -> base_noise, frame_ready suppressed
DARK_SETTLE_MS          = 12    # dark-frame settle window (one frame ~109 us)

LAT_PL_PERIOD_NS = 1_000_000_000 / 520_000_000   # 1.92308 ns/tick @ 520 MHz
RESULT_BYTES     = 25 * 2
VDMA_HSIZE_BYTES = 512
VDMA_VSIZE_LINES = 512
MIN_FRAME_MS     = 0.05   # 50 us: well under the ~109 us 8 px/beat frame

print('\nAll handles and constants defined.')


#### Step 4 - Coordinate programming


In [ ]:
def write_coordinates(x_offset=0, y_offset=0):
    for q in range(NUM_QUBITS):
        qx = calc_qx(q, x_offset)
        qy = calc_qy(q, y_offset) + 1   # +1 centres qubit row in 3-row ROI window
        coord_ip.write(q*8+0, int(qx));  coord_ip.read(q*8+0)   # B-channel fence
        coord_ip.write(q*8+4, int(qy));  coord_ip.read(q*8+4)
    print(f'  Wrote {NUM_QUBITS} coords  offset=(+{x_offset},+{y_offset})')

def verify_coordinates(x_offset=0, y_offset=0, spot_check=None):
    qs = spot_check if spot_check is not None else range(NUM_QUBITS)
    errors = 0
    for q in qs:
        exp_x = calc_qx(q, x_offset)
        exp_y = calc_qy(q, y_offset) + 1
        got_x = coord_ip.read(q*8+0) & 0x1FF
        got_y = coord_ip.read(q*8+4) & 0x1FF
        if got_x != exp_x or got_y != exp_y:
            print(f'  ERROR q{q}: expected ({exp_x},{exp_y}) got ({got_x},{got_y})')
            errors += 1
    if errors == 0:
        print(f'  Verified {len(list(qs))} coordinates -- all PASS ok')
    else:
        raise RuntimeError(f'Coordinate readback: {errors} mismatches')

print('Writing qubit coordinates ...')
write_coordinates()
verify_coordinates(spot_check=[0, 1, 9, 50, 99])
print('lut_valid asserted -- FPGA pixel gate open.')


#### Step 4b - Dark-mode + Gaussian-threshold control registers
`qubit_lookup_axi` exposes two software control registers on the coordinate
AXI4-Lite slave: `0x320` bit 0 = `dark_mode`, `0x324` `[15:0]` = the runtime
Gaussian decision threshold. v8 never touched either; v9 pulls them out as
named parameters with write + readback helpers, mirroring `program_dark_mode()`
and `program_gauss_threshold()` in `testbench_dut_extended.sv`.

In [ ]:
"""
Thin write/verify helpers for the two qubit_lookup_axi control registers.
Both share the coordinate AXI4-Lite slave (coord_ip). Each write is followed
by a read ('B-channel fence') so the AXI write response is retired before we
move on, then a readback check -- exactly mirroring write_coordinates() and
the program_dark_mode()/program_gauss_threshold() tasks in the RTL testbench.
"""

def write_dark_mode(mode, verify=True):
    """mode: 0 = active readout, 1 = dark / noise-baseline capture."""
    val = 1 if mode else 0
    coord_ip.write(CTRL_DARK_MODE, val)
    coord_ip.read(CTRL_DARK_MODE)               # B-channel fence
    time.sleep(0.001)                           # let the 300->520 2-FF sync settle
    if verify:
        rb = coord_ip.read(CTRL_DARK_MODE) & 0x1
        if rb != val:
            raise RuntimeError(f'dark_mode readback={rb}, expected {val}')
        print(f'  dark_mode = {val}  ({"CAPTURE" if val else "ACTIVE"})  readback OK')
    return val

def read_dark_mode():
    return coord_ip.read(CTRL_DARK_MODE) & 0x1

def write_gauss_threshold(thresh, verify=True):
    """thresh: 16-bit unsigned Gaussian decision threshold."""
    t = int(thresh) & 0xFFFF
    coord_ip.write(CTRL_GAUSS_THRESH, t)
    coord_ip.read(CTRL_GAUSS_THRESH)            # B-channel fence
    time.sleep(0.001)
    if verify:
        rb = coord_ip.read(CTRL_GAUSS_THRESH) & 0xFFFF
        if rb != t:
            raise RuntimeError(f'gaussian_threshold readback={rb}, expected {t}')
        print(f'  gaussian_threshold = {t}  readback OK')
    return t

def read_gauss_threshold():
    return coord_ip.read(CTRL_GAUSS_THRESH) & 0xFFFF

# Bring the pipeline to a known state: active readout, default threshold.
write_dark_mode(DARK_MODE_ACTIVE)
write_gauss_threshold(GAUSS_THRESHOLD_DEFAULT)
print(f'Control registers initialised: active mode, threshold = {GAUSS_THRESHOLD_DEFAULT}.')

#### Step 5 - DMA buffer allocation


In [ ]:
# result_buf non-cacheable (50 B; uncached read is free, saves the
# 60 us invalidate every frame). frame_buf is kept ONLY for run_dark_capture
# and as a fallback -- the hot path uses PINNED[name] (see Step 10c).
frame_buf  = allocate(shape=(512, 512), dtype=np.uint8)
try:
    result_buf = allocate(shape=(25,), dtype=np.uint16, cacheable=False)
    _result_buf_cacheable = False
except TypeError:
    # older PYNQ without the cacheable kwarg -- fall back, the invalidate
    # in run_frame() will then be necessary on every frame.
    result_buf = allocate(shape=(25,), dtype=np.uint16)
    _result_buf_cacheable = True
    print('WARN: result_buf is cacheable ; per-frame invalidate kept.')
print(f'frame_buf   PA=0x{frame_buf.physical_address:010X}   {frame_buf.nbytes} B')
print(f'result_buf  PA=0x{result_buf.physical_address:010X}   {result_buf.nbytes} B  '
      f'(cacheable={_result_buf_cacheable})')
assert frame_buf.physical_address  < 0x1_0000_0000
assert result_buf.physical_address < 0x1_0000_0000
print('Both buffers within 32-bit address space ok')


#### Step 6 - VDMA helpers (64-bit stream)


In [ ]:
def vdma_reset(timeout_ms=500):
    vdma.write(VDMA_CR, 0x4)
    t0 = time.time()
    while vdma.read(VDMA_CR) & 0x4:
        if (time.time()-t0)*1000 > timeout_ms: raise TimeoutError('VDMA reset stuck')
        time.sleep(0.001)

def vdma_stop(timeout_ms=10):
    vdma.write(VDMA_CR, 0)
    t0 = time.time()
    while not (vdma.read(VDMA_SR) & 1):
        if (time.time()-t0)*1000 > timeout_ms: break
        time.sleep(0.0001)

def vdma_send_frame(phys_addr):
    """
    Start one MM2S transfer in park mode.

    64-bit VDMA: HSIZE = 512 bytes/line (always bytes, not pixels).
    Stream width is 64-bit (8 px/beat); HSIZE unchanged, beats/line = 64.
    """
    pa = int(phys_addr)
    assert pa < 0x1_0000_0000
    vdma.write(VDMA_CR,     0x1)              # RS=1, park mode
    vdma.write(VDMA_ADDR1,  pa)
    vdma.write(VDMA_ADDR2,  pa)               # belt-and-suspenders: same PA
    vdma.write(VDMA_ADDR3,  pa)
    vdma.write(VDMA_HSIZE,  VDMA_HSIZE_BYTES)
    vdma.write(VDMA_STRIDE, VDMA_HSIZE_BYTES)
    vdma.write(VDMA_VSIZE,  VDMA_VSIZE_LINES) # LAST write -- triggers transfer

print('VDMA helpers defined.')
print(f'  VDMACR=0x{vdma.read(VDMA_CR):08X}  VDMASR=0x{vdma.read(VDMA_SR):08X}')


#### Step 7 - DMA S2MM helpers


In [ ]:
def dma_reset(timeout_ms=200):
    dma.write(S2MM_CR, 0x4)
    t0 = time.time()
    while dma.read(S2MM_CR) & 0x4:
        if (time.time()-t0)*1000 > timeout_ms: raise TimeoutError('DMA reset stuck')
        time.sleep(0.001)

def dma_recv_start(phys_addr, length=RESULT_BYTES):
    pa = int(phys_addr)
    assert pa < 0x1_0000_0000
    dma.write(S2MM_CR,  0x1)
    dma.write(S2MM_DA,  pa)
    dma.write(S2MM_LEN, length)   # arms -- DMA waits for tlast

def dma_recv_wait(timeout_ms=2000):
    """Legacy sleep-based poll (200 us granularity). Kept for fallback;
       hot path now uses dma_recv_wait_busy()."""
    t0 = time.perf_counter_ns()
    deadline = t0 + int(timeout_ms * 1_000_000)
    while True:
        sr = dma.read(S2MM_SR)
        if sr & (1 << 12):
            t_ioc = time.perf_counter_ns()
            dma.write(S2MM_SR, sr)
            return t_ioc
        if sr & 0x70:
            raise RuntimeError(f'DMA S2MM error: DMASR=0x{sr:08X}')
        if time.perf_counter_ns() > deadline:
            raise TimeoutError(f'DMA S2MM timeout -- DMASR=0x{dma.read(S2MM_SR):08X}')
        time.sleep(0.0002)

def dma_recv_wait_busy(timeout_ms=10):
    """ busy-wait for S2MM IOC. Drops the ~100 us mean poll gap to
       ~5 us (single MMIO read latency). CPU pegged at 100% for the ~110 us
       the FPGA pipeline takes -- that IS the latency we are minimising.
       Returns the perf_counter_ns timestamp at the cycle IOC was seen."""
    deadline = time.perf_counter_ns() + int(timeout_ms * 1_000_000)
    while True:
        sr = dma.read(S2MM_SR)
        if sr & (1 << 12):
            t_ioc = time.perf_counter_ns()
            dma.write(S2MM_SR, sr)
            return t_ioc
        if sr & 0x70:
            raise RuntimeError(f'DMA S2MM error: DMASR=0x{sr:08X}')
        if time.perf_counter_ns() > deadline:
            raise TimeoutError(f'DMA S2MM timeout -- DMASR=0x{dma.read(S2MM_SR):08X}')

def _drain_output_fifo():
    """Clear stale beats left in output CDC FIFO by the ghost frame.
        kept for run_dark_capture(); the lean hot path no longer
       calls this -- it is provably redundant after vdma_stop() halts
       VDMA upstream of the PL pipeline (verify by sweeping frames
       and checking decode_results() never raises 'Ghost frame ...')."""
    for _ in range(2):
        dma_reset()
        dma_recv_start(result_buf.physical_address)
        t0 = time.time()
        found = False
        while (time.time()-t0)*1000 < 0.5:
            sr = dma.read(S2MM_SR)
            if sr & (1 << 12):
                dma.write(S2MM_SR, sr); found = True; break
            if sr & 0x70: dma_reset(); return
            time.sleep(0.00005)
        dma_reset()
        if not found: break

print('DMA helpers defined (dma_recv_wait_busy available).')
print(f' S2MM_DMASR=0x{dma.read(S2MM_SR):08X}')


#### Step 8- Result decoder


In [ ]:
# vectorised decoder. Same input/output as v9, same RuntimeError on
# ghost frame -- but numpy broadcast instead of a 25 x 4 = 100-iter Python
# loop. Drops decode from ~400 us to ~5 us.
_EXPECTED_BASE_IDS = np.arange(25, dtype=np.uint16) * 4
_LANE_MASK         = np.array([1, 2, 4, 8], dtype=np.uint16)

def decode_results(buf_uint16):
    """25 AXI-S beats -> 100-element bool array.
       Beat format: tdata[15:0] = { 5b0, base_id[6:0], state[3:0] }
       Raises RuntimeError on base_id mismatch (partial ghost frame)."""
    buf = np.asarray(buf_uint16, dtype=np.uint16)
    base_ids = (buf >> 4) & 0x7F
    if not np.array_equal(base_ids, _EXPECTED_BASE_IDS):
        bad = int(np.argmax(base_ids != _EXPECTED_BASE_IDS))
        raise RuntimeError(
            f'Ghost frame -- beat {bad}: '
            f'expected base_id={int(_EXPECTED_BASE_IDS[bad])}, '
            f'got {int(base_ids[bad])} (0x{int(buf[bad]):04X})')
    bits = (buf[:, None] & _LANE_MASK) != 0   # (25, 4) bool matrix
    return bits.ravel()[:100]                 # drop padding past q99

# Self-test (unchanged from v9 -- byte-identical contract)
buf = np.array([(i*4 << 4) for i in range(25)], dtype=np.uint16)
buf[0]  = (0  << 4) | 0b1001
buf[24] = (96 << 4) | 0b1111
s = decode_results(buf)
assert s[0] and not s[1] and s[3] and all(s[96:100]) and not any(s[4:96])
print('Decoder self-test PASSED ok (vectorised)')


#### Step 9 - latency_capture- reader
Reads all 15 FPGA-side registers in a frame_seq-guarded coherent window.
Timestamps are quasi-static (stable ~109 us between frames) so direct cross-domain
reads are safe; the seq guard catches the rare frame-boundary colisons


In [ ]:
LatSnap = namedtuple('LatSnap', [
    'frame_seq',
    'ts_sof', 'ts_match', 'ts_roi_wr', 'ts_frame_ready',
    'ts_first_result', 'ts_last_result', 'ts_frame_done',
    'delta_pipe', 'delta_match', 'delta_roi',
    'delta_storage', 'delta_readout', 'delta_filter',
])

def read_latency_snapshot(max_retries=3):
    """
    Read all latency_capture registers coherently.
    frame_seq is read before and after the 13 data registers.
    If it changed (new frame completed mid-read) the snapshot is retried.
    """
    for _ in range(max_retries):
        seq0       = lat_ip.read(LAT_FRAME_SEQ)
        ts_sof     = lat_ip.read(LAT_TS_SOF)
        ts_match   = lat_ip.read(LAT_TS_MATCH)
        ts_roi     = lat_ip.read(LAT_TS_ROI_WR)
        ts_ready   = lat_ip.read(LAT_TS_READY)
        ts_first_r = lat_ip.read(LAT_TS_FIRST_R)
        ts_last_r  = lat_ip.read(LAT_TS_LAST_R)
        ts_fdone   = lat_ip.read(LAT_TS_FDONE)
        d_pipe     = lat_ip.read(LAT_D_PIPE)
        d_match    = lat_ip.read(LAT_D_MATCH)
        d_roi      = lat_ip.read(LAT_D_ROI)
        d_storage  = lat_ip.read(LAT_D_STORAGE)
        d_readout  = lat_ip.read(LAT_D_READOUT)
        d_filter   = lat_ip.read(LAT_D_FILTER)
        seq1       = lat_ip.read(LAT_FRAME_SEQ)
        if seq0 == seq1:
            return LatSnap(seq0, ts_sof, ts_match, ts_roi, ts_ready,
                           ts_first_r, ts_last_r, ts_fdone,
                           d_pipe, d_match, d_roi, d_storage, d_readout, d_filter)
    raise RuntimeError(f'Latency snapshot incoherent after {max_retries} retries')

SWTiming = namedtuple('SWTiming', [
    'cache_flush_ns', 'dma_arm_ns', 'vdma_start_ns',
    'fpga_wall_ns', 'vdma_stop_ns', 'cache_inv_ns', 'decode_ns', 'total_ns',
])

def latency_report(snap, sw, label=''):
    """
    Print a formatted per-frame latency breakdown.

    FPGA PL section (latency_capture registers, 520 MHz resolution):
      delta_match   = SOF -> first coord_matcher hit
      delta_roi     = first match -> first ROI write  (extractor latency)
      delta_storage = frame_done -> frame_ready       (ping-pong CDC)
      delta_readout = frame_ready -> last result      (read_streamer + gaussian)
      delta_filter  = first -> last result            (gaussian bank throughput)
      delta_pipe    = SOF -> last result              TOTAL PL pipeline

    CPU/SW section (perf_counter_ns, ~50 ns OS resolution):
      cache_flush   = sync_to_device() or flush()
      dma_arm       = vdma_stop + drain + dma_reset + invalidate + dma_recv_start
      vdma_start    = vdma_send_frame() overhead
      fpga_wall     = VDMA_VSIZE write -> DMA IOC   (includes FPGA + CDC + poll)
      ioc_overhead  = fpga_wall - delta_pipe_ns     (OS scheduling + DMA noise)
      vdma_stop     = RS=0 -> Halted
      cache_inv     = result_buf.invalidate()
      decode        = decode_results()
      total_sw      = cache_flush_start -> decode_end
    """
    P  = LAT_PL_PERIOD_NS
    def cy(c): return f'{c:8d} cy  {c*P:9.2f} ns  {c*P/1000:7.3f} us'
    def ns(n): return f'           {n:9.1f} ns  {n/1000:7.3f} us'
    pipe_ns  = snap.delta_pipe * P
    overhead = sw.fpga_wall_ns - pipe_ns

    print()
    print(f'  +-- LATENCY  frame={label}  seq={snap.frame_seq}  clock=520MHz ({P:.4f} ns/tick) ----+')
    print(f'  | FPGA PL pipeline (hardware deterministic)')
    print(f'  |   SOF -> first match     {cy(snap.delta_match)}')
    print(f'  |   match -> first ROI     {cy(snap.delta_roi)}')
    print(f'  |   fdone -> frm_ready     {cy(snap.delta_storage)}')
    print(f'  |   ready -> last result   {cy(snap.delta_readout)}')
    print(f'  |   first -> last result   {cy(snap.delta_filter)}')
    print(f'  |   TOTAL PL pipeline      {cy(snap.delta_pipe)}')
    print(f'  | CPU / SW layers (OS-jittered- I think this part is undeterminstic to some sense)')
    print(f'  |   cache flush            {ns(sw.cache_flush_ns)}')
    print(f'  |   DMA arm + drain        {ns(sw.dma_arm_ns)}')
    print(f'  |   VDMA start             {ns(sw.vdma_start_ns)}')
    print(f'  |   FPGA wall (poll+CDC)   {ns(sw.fpga_wall_ns)}')
    print(f'  |   IOC overhead           {ns(overhead)}   <- wall minus PL pipe')
    print(f'  |   VDMA stop              {ns(sw.vdma_stop_ns)}')
    print(f'  |   cache invalidate       {ns(sw.cache_inv_ns)}')
    print(f'  |   decode                 {ns(sw.decode_ns)}')
    print(f'  |   TOTAL end-to-end       {ns(sw.total_ns)}')
    print(f'  +--------------------------------------------------------------------------+')

print('Latency reader and SWTiming defined.')


#### Step 10 - run_frame() sequencer with full timing instrumentation
Every software stage is individually timestamped. The FPGA snapshot is read in the
inter-frame idle window after vdma_stop() confirms the ghost frame won't
interfere with the 15 MMIO reads.


In [ ]:
def run_frame(name, verbose=True):
    """
    one-shot frame run with ns-resolution SW timing and FPGA latency
    capture. Lean hot path: no per-frame copy, no per-frame flush, no drain,
    no zero-fill, no invalidate (if result_buf is non-cacheable), busy-wait
    IOC.

    Returns  (states, sw, hw)
      states  np.ndarray[bool,100]  qubit state vector
      sw      SWTiming              ns-resolution SW layer breakdown
      hw      LatSnap               FPGA cycle-accurate snapshot

    Sequencing (v10 vs v9 deltas marked):
      [T0]  pick frame_pa = PINNED[name].physical_address       (v10: no copy/flush)
      [T1]  -- bookkeeping only --                              (was 412 us)
      [T2]  vdma_stop()                                         (busy-wait, ~5 us)
      [T3]  dma_recv_start(result_buf.PA)                       (was drain+reset+zero+inv+arm)
      [T4]  vdma_send_frame(frame_pa)
      [T5]  dma_recv_wait_busy() -> t_ioc                       (v10: busy poll, no sleep)
      [T6]  vdma_stop()
      [T7]  read_latency_snapshot()
      [T8]  decode_results() (no invalidate if non-cacheable)   (v10: vectorised)
    """
    MAX_RETRIES = 2

    # --- T0  pick the pinned, pre-flushed CMA buffer for this pattern ---
    pin = PINNED.get(name) if 'PINNED' in globals() else None
    if pin is not None:
        frame_pa = pin.physical_address
    else:
        # Fallback path: behave like v9 (one-off copy + flush). Used for
        # patterns that were not pinned at setup (e.g. ad-hoc dark frames).
        raw = FRAMES.get(name) if 'FRAMES' in globals() else None
        if raw is None:
            raw = np.fromfile(f'frames/{name}.bin', dtype=np.uint8).reshape(512, 512)
        frame_buf[:] = raw
        if hasattr(frame_buf, 'sync_to_device'): frame_buf.sync_to_device()
        else:                                    frame_buf.flush()
        frame_pa = frame_buf.physical_address

    for attempt in range(1 + MAX_RETRIES):

        # T1  cache flush -- 0 in v10 hot path (pinned + pre-flushed)
        t_cf0 = t_cf1 = time.perf_counter_ns()

        # T2+T3  lean arm:  vdma_stop (busy-wait) + dma_recv_start (3 MMIO)
        t_arm0 = time.perf_counter_ns()
        vdma_stop()
        # NB: drop _drain_output_fifo, dma_reset, zero-fill, invalidate.
        # Safe iff (a) ghost frames cannot reach S2MM after vdma_stop and
        # (b) result_buf is non-cacheable. If decode_results() raises
        # 'Ghost frame', restore the drain/reset for that path.
        if _result_buf_cacheable:
            result_buf.invalidate()
        dma_recv_start(result_buf.physical_address)
        t_arm1 = time.perf_counter_ns()

        # T4  pixels start flowing
        t_vs0 = time.perf_counter_ns()
        vdma_send_frame(frame_pa)
        t_vs1 = time.perf_counter_ns()

        # T5  busy-wait for DMA IOC -- FPGA pipeline completes here
        t_ioc = dma_recv_wait_busy(timeout_ms=10)

        # T6  stop VDMA before next frame can be emitted in park mode
        t_stop0 = time.perf_counter_ns()
        vdma_stop()
        t_stop1 = time.perf_counter_ns()

        wall_ns = t_ioc - t_vs1   # T5 - T4

        # Elapsed guard (catches stale FIFO data on a misconfigured run)
        if wall_ns < int(MIN_FRAME_MS * 1_000_000):
            if attempt < MAX_RETRIES:
                if verbose: print(f'  [retry {attempt+1}] stale ({wall_ns/1e6:.3f} ms < {MIN_FRAME_MS} ms)')
                continue
            raise RuntimeError(f"run_frame('{name}'): persistent stale data")

        # T7  FPGA latency snapshot (15 MMIO reads, coherent under seq guard)
        hw = read_latency_snapshot()

        # T8  decode -- skip invalidate if result_buf is non-cacheable
        t_inv0 = time.perf_counter_ns()
        if _result_buf_cacheable:
            result_buf.invalidate()
        t_inv1 = time.perf_counter_ns()
        t_dec0 = time.perf_counter_ns()
        try:
            states = decode_results(np.array(result_buf))
        except RuntimeError:
            if attempt < MAX_RETRIES:
                if verbose: print(f'  [retry {attempt+1}] partial ghost')
                continue
            raise
        t_dec1 = time.perf_counter_ns()

        sw = SWTiming(
            cache_flush_ns = t_cf1  - t_cf0,
            dma_arm_ns     = t_arm1 - t_arm0,
            vdma_start_ns  = t_vs1  - t_vs0,
            fpga_wall_ns   = wall_ns,
            vdma_stop_ns   = t_stop1 - t_stop0,
            cache_inv_ns   = t_inv1  - t_inv0,
            decode_ns      = t_dec1  - t_dec0,
            total_ns       = t_dec1  - t_cf0,
        )

        if verbose:
            pl_us = hw.delta_pipe * LAT_PL_PERIOD_NS / 1000
            print(f'  {name:<16} '
                  f'PL_pipe={pl_us:.1f}us  '
                  f'wall={wall_ns/1e3:.1f}us  '
                  f'total={sw.total_ns/1e6:.3f}ms  '
                  f'Rydberg={int(np.sum(states))}/100')

        return states, sw, hw

    raise RuntimeError(f"run_frame('{name}'): exhausted retries")

print('run_frame() defined (v10 lean hot path).')


#### Step 10b - run_dark_capture(): noise-baseline capture
With `dark_mode = 1`, `roi_storage` routes every ROI into the `base_noise`
banks and suppresses `o_frame_ready`, so `read_streamer` never starts and no
result beats reach the S2MM DMA. `run_dark_capture()` injects one calibration
frame and proves the suppression two ways: S2MM raises no IOC for the whole
window, and `latency_capture` shows the pipeline *did* run the frame
(`frame_seq` and `ts_first_roi_wr` advanced) while the ping-pong swap did *not*
happen (`ts_frame_ready` unchanged).

`capture_baseline()` wraps a full enter/capture/exit cycle; `reset_noise_baseline()`
captures the all-zero frame so the `base_noise` banks go back to zero (the only
way short of reloading the bitstream, since the banks have no logic reset).

In [ ]:
def run_dark_capture(name, settle_ms=None, verbose=True):
    """
    Inject ONE dark / calibration frame while dark_mode is already = 1.

    In dark mode roi_storage routes every ROI into the base_noise banks and
    SUPPRESSES o_frame_ready -- so read_streamer never starts and NO result
    beats reach the S2MM DMA. This routine proves the suppression:

      (1) S2MM never raises IOC during the whole capture window.
      (2) latency_capture confirms the pipeline DID run the frame
          (frame_seq advanced, ts_first_roi_wr moved) but the ping-pong
          swap did NOT happen (ts_frame_ready unchanged).

    Returns (ok, snap_before, snap_after).
    """
    if settle_ms is None:
        settle_ms = DARK_SETTLE_MS
    if read_dark_mode() != 1:
        raise RuntimeError('run_dark_capture requires dark_mode = 1')

    raw = np.fromfile(f'frames/{name}.bin', dtype=np.uint8).reshape(512, 512)

    # clean slate: stop VDMA, drain any stale output beats, reset S2MM
    vdma_stop()
    _drain_output_fifo()
    dma_reset()
    time.sleep(0.001)

    # pipeline state BEFORE the capture (idle window -> coherent snapshot)
    snap0 = read_latency_snapshot()

    # load + cache-flush the dark frame
    frame_buf[:] = raw
    if hasattr(frame_buf, 'sync_to_device'): frame_buf.sync_to_device()
    else: frame_buf.flush()

    # arm S2MM purely so we can detect the (expected) ABSENCE of completion
    result_buf[:] = 0
    result_buf.invalidate()
    dma_recv_start(result_buf.physical_address)

    # stream the dark frame into the pipeline
    vdma_send_frame(frame_buf.physical_address)

    # watch for any spurious IOC across the whole settle window
    spurious_ioc = False
    deadline = time.perf_counter_ns() + int(settle_ms * 1_000_000)
    while time.perf_counter_ns() < deadline:
        sr = dma.read(S2MM_SR)
        if sr & (1 << 12):
            spurious_ioc = True
            dma.write(S2MM_SR, sr)            # W1C, keep watching
        elif sr & 0x70:
            break                            # DMA error -- stop, reported below
        time.sleep(0.0002)

    vdma_stop()
    dma_reset()
    time.sleep(0.001)

    # pipeline state AFTER (idle window -> coherent snapshot)
    snap1 = read_latency_snapshot()

    seq_adv   = (snap1.frame_seq - snap0.frame_seq) & 0xFFFFFFFF
    roi_moved = snap1.ts_roi_wr      != snap0.ts_roi_wr
    rdy_moved = snap1.ts_frame_ready != snap0.ts_frame_ready

    ok = (not spurious_ioc) and (seq_adv > 0) and roi_moved and (not rdy_moved)

    if verbose:
        print(f'  dark capture "{name}":')
        print(f'    frame_seq advanced     : +{seq_adv}'
              f'   {"OK" if seq_adv > 0 else "FAIL (frame not processed)"}')
        print(f'    ts_first_roi_wr moved  : {roi_moved}'
              f'   {"OK (ROIs -> base_noise banks)" if roi_moved else "FAIL"}')
        print(f'    ts_frame_ready moved   : {rdy_moved}'
              f'   {"OK (ping-pong swap suppressed)" if not rdy_moved else "FAIL (frame_ready fired!)"}')
        print(f'    spurious S2MM IOC      : {spurious_ioc}'
              f'   {"OK (no result beats)" if not spurious_ioc else "FAIL (output during dark mode!)"}')
        print(f'    -> dark capture {"PASS" if ok else "*** FAIL ***"}')
    return ok, snap0, snap1


def capture_baseline(name):
    """Full dark-capture cycle: enter dark mode, capture <name>, return to active."""
    write_dark_mode(DARK_MODE_CAPTURE, verify=False)
    ok, _, _ = run_dark_capture(name, verbose=True)
    write_dark_mode(DARK_MODE_ACTIVE, verify=False)
    if not ok:
        raise RuntimeError(f'capture_baseline("{name}") failed dark-mode checks')
    return ok


def reset_noise_baseline():
    """Capture the all-zero dark frame -> base_noise banks = 0 -> subtraction is a no-op."""
    print('  resetting noise baseline (capture dark_zero) ...')
    return capture_baseline('dark_zero')


print('run_dark_capture(), capture_baseline(), reset_noise_baseline() defined.')

#### Step 10c - Pre-load frames into RAM (lean hot path)
Loads every frame that goes through `run_frame()` into a RAM-resident dict so
the latency-critical hot path never touches the disk - one of the avoidable
software-jitter sources. Dark / calibration frames stay on-demand
(`run_dark_capture()` is one-time setup, not in the hot path).

In [ ]:
"""
 lean hot path: one PYNQ-allocated, pre-flushed buffer per pattern.

Why: in v9 every run_frame() did `frame_buf[:] = raw; sync_to_device()` which
cost ~412 us (and a 278 us stddev) on every call. Here we allocate one
contiguous CMA buffer per pattern at setup time, fill + flush it once, and
the hot path only passes its physical_address to vdma_send_frame() -- no
copy, no flush.

Memory cost: ~7 * 256 KB = ~1.75 MB of CMA. Trivial on a 4 GB ZynqMP.

FRAMES (numpy in normal RAM) is retained for run_dark_capture() and as a
disk-fallback for run_frame() when a name has no pinned buffer.
"""
_HOT_FRAMES = list(patterns.keys()) + ['verify_active']
FRAMES = {}
PINNED = {}
for _name in _HOT_FRAMES:
    _p = f'frames/{_name}.bin'
    if not os.path.exists(_p):
        continue
    _raw = np.fromfile(_p, dtype=np.uint8).reshape(512, 512)
    FRAMES[_name] = _raw                           # plain RAM copy (fallback / dark capture)
    _buf = allocate(shape=(512, 512), dtype=np.uint8)
    _buf[:] = _raw
    if hasattr(_buf, 'sync_to_device'): _buf.sync_to_device()   # one-time flush
    else:                               _buf.flush()
    PINNED[_name] = _buf
print(f'Pre-loaded {len(FRAMES)} frames into RAM and {len(PINNED)} pinned/pre-flushed CMA buffers.')
print(f'  Pinned names: {list(PINNED)}')
print('run_frame() hot path now skips the per-frame copy + sync_to_device entirely.')


#### Step 11 - Smoke test


In [ ]:
print('Smoke test: all_ground')
states_sg, sw_sg, hw_sg = run_frame('all_ground', verbose=True)

assert int(np.sum(states_sg)) == 0, f'Expected 0 Rydberg, got {int(np.sum(states_sg))}'
assert sw_sg.fpga_wall_ns > MIN_FRAME_MS * 1e6, 'Wall too short -- stale FIFO?'

print('\nSmoke test PASSED ok')
latency_report(hw_sg, sw_sg, label='all_ground')


#### Step 12 - SETUP PHASE: one-time base-noise capture
**Dark Mode ON = setup phase.** Capture the experiment's static background
(dark current, fixed-pattern read noise, stray light) once, into the sticky
`base_noise` BRAM banks inside `roi_storage`. They persist until overwritten or
the bitstream is reloaded - a true one-time calibration.

**Dark Mode OFF = the real experiment.** From here on `read_streamer` subtracts
this baseline from every ROI, per pixel (`max(0, roi - noise)`), raising the
signal-to-noise ratio of the Rydberg readout.

On the real instrument, replace `BASE_NOISE_FRAME` below with a real
closed-shutter acquisition pushed through the same VDMA path.

In [ ]:
# ------------------------------------------------------------------ 
#  SETUP PHASE  --  run ONCE, before the experiment
# ------------------------------------------------------------------ 
#  Dark Mode ON   -> capture the experiment's base noise into the sticky
#                    base_noise BRAM banks inside roi_storage
#  Dark Mode OFF  -> the real experiment; per-pixel noise subtraction is
#                    live from here on  (read_streamer: max(0, roi - noise))
# --------------------------------------------------------------------
#  On the real instrument, replace BASE_NOISE_FRAME with a real
#  closed-shutter acquisition pushed through the same VDMA path.
# ------------------------------------------------------------------ 
BASE_NOISE_FRAME = 'dark_pedestal'

print('=== SETUP PHASE: one-time base-noise capture ===')
print(f'  base-noise frame : {BASE_NOISE_FRAME}')
capture_baseline(BASE_NOISE_FRAME)        # dark ON -> capture (sticky) -> dark OFF
assert read_dark_mode() == 0, 'dark_mode must be OFF after the setup phase'
print()
print('  base_noise banks loaded and STICKY (persist until overwritten / reload).')
print('  dark_mode = 0  ->  noise subtraction is LIVE for every experiment frame.')
print('  >> proceed to Step 13 (EXPERIMENT) <<')

#### Step 13 - EXPERIMENT: run all frames (noise subtraction live)
The real run. `dark_mode` is OFF, so `read_streamer` subtracts the base-noise
baseline captured in Step 12 from every ROI, per pixel. Each frame prints its
cycle-accurate `latency_capture` breakdown; the pixel images are plotted with
the per-qubit ROI boxes coloured by the hardware decision - same view as v8.

In [ ]:
# --- EXPERIMENT runs with the STICKY base-noise baseline from the SETUP PHASE.
# --- Assert active mode + default threshold only; the base_noise banks are
# --- intentionally left exactly as captured in Step 12 -- do NOT reset them.
# --- (If you ran the optional self-test in Step 15, re-run Step 12 first.)
assert read_dark_mode() == 0, 'run the SETUP PHASE (Step 12) first -- dark_mode must be OFF'
write_gauss_threshold(GAUSS_THRESHOLD_DEFAULT)

from matplotlib.gridspec import GridSpec

ROI_HALF    = 1
results_log = {}
hw_log      = {}
sw_log      = {}

fig = plt.figure(figsize=(15, 10), facecolor='#080810')
plt.suptitle('ZCU216 Qubit Readout v9 -- noise subtraction live + latency capture',
             color='white', fontsize=13, y=1.01)
gs = GridSpec(2, 3, figure=fig, hspace=0.15, wspace=0.06)

for idx, (name, excited) in enumerate(patterns.items()):

    states, sw, hw = run_frame(name, verbose=True)
    latency_report(hw, sw, label=name)

    expected = np.array(excited)
    errors   = int(np.sum(states != expected))
    results_log[name] = dict(states=states, expected=expected, errors=errors)
    hw_log[name] = hw
    sw_log[name] = sw

    ax  = fig.add_subplot(gs[idx // 3, idx % 3])
    img = np.fromfile(f'frames/{name}.bin', dtype=np.uint8).reshape(512, 512)
    ax.imshow(img, cmap='inferno', origin='upper',
              norm=PowerNorm(gamma=0.4, vmin=0, vmax=255), interpolation='nearest')
    for q in range(NUM_QUBITS):
        qx, qy = coords[q]
        hw_r   = bool(states[q])
        sw_r   = bool(excited[q])
        ax.add_patch(patches.Rectangle(
            (qx-ROI_HALF-0.5, qy-ROI_HALF-0.5), 2*ROI_HALF+1, 2*ROI_HALF+1,
            lw=0.6, edgecolor='#00e5ff' if hw_r else '#ff6e40', facecolor='none'))
        if hw_r != sw_r:
            ax.plot(qx, qy, 'rx', markersize=6, markeredgewidth=1.2)
    pl_us  = hw.delta_pipe * LAT_PL_PERIOD_NS / 1000
    status = f"{100-errors}/100 ok" if errors == 0 else f"{100-errors}/100 FAIL{errors}"
    ax.set_title(f'{name}  |  {int(np.sum(states))}R  |  {status}  |  PL={pl_us:.0f}us  wall={sw.fpga_wall_ns/1e3:.0f}us',
                 color='white', fontsize=7.5, pad=3)
    ax.axis('off')

plt.tight_layout()
plt.savefig('fpga_results_v9.png', dpi=130, bbox_inches='tight', facecolor='#080810')
plt.show()

print(f"\n{'Frame':<16}  {'R/100':>6}  {'Err':>4}  {'PL us':>7}  {'Wall us':>8}  {'IOC oh us':>10}  {'Total ms':>9}")
print('-' * 75)
all_pass = True
for name in patterns:
    r  = results_log[name]
    hw = hw_log[name]
    sw = sw_log[name]
    pl_us = hw.delta_pipe * LAT_PL_PERIOD_NS / 1000
    wa_us = sw.fpga_wall_ns / 1000
    oh_us = (sw.fpga_wall_ns - hw.delta_pipe * LAT_PL_PERIOD_NS) / 1000
    ok    = 'ok' if r['errors'] == 0 else 'FAIL'
    if r['errors']: all_pass = False
    print(f"  {name:<14}  {int(np.sum(r['states'])):>6}  {r['errors']:>4}  {pl_us:>7.1f}  {wa_us:>8.1f}  {oh_us:>10.1f}  {sw.total_ns/1e6:>9.3f}  {ok}")
print('-' * 75)
print(f"  {'ALL PASS' if all_pass else 'SOME FAILED'}")


#### Step 14 - Determinism analysis
Runs the same frame N times. `delta_pipe` (FPGA hardware) should be bit-exact
every run. All jitter is in the CPU/OS layers. This directly demonstrates
what is and isn't deterministic in the system.


In [ ]:
N_RUNS     = 20     # increase to 50+ for publication histograms
TEST_FRAME = 'all_rydberg'

pl_cyc_log = []
wall_us_log    = []
oh_us_log      = []
total_ms_log   = []
cf_us_log      = []

print(f'Running {N_RUNS}x {TEST_FRAME} for determinism analysis ...')
for i in range(N_RUNS):
    _, sw, hw = run_frame(TEST_FRAME, verbose=False)
    pl_cyc_log.append(hw.delta_pipe)
    wall_us_log.append(sw.fpga_wall_ns / 1000)
    oh_us_log.append((sw.fpga_wall_ns - hw.delta_pipe * LAT_PL_PERIOD_NS) / 1000)
    total_ms_log.append(sw.total_ns / 1e6)
    cf_us_log.append(sw.cache_flush_ns / 1000)
    if (i+1) % 5 == 0: print(f'  {i+1}/{N_RUNS}')

pl_arr   = np.array(pl_cyc_log, dtype=np.int64)
wall_arr = np.array(wall_us_log)
oh_arr   = np.array(oh_us_log)
tot_arr  = np.array(total_ms_log)
cf_arr   = np.array(cf_us_log)

print()
print('-- FPGA PL delta_pipe (deterministic hardware) ----------------------')
spread_cy = pl_arr.max() - pl_arr.min()
print(f'   min={pl_arr.min()}  max={pl_arr.max()}  spread={spread_cy} cycles ({spread_cy*LAT_PL_PERIOD_NS:.1f} ns)')
print(f'   -> {"FULLY DETERMINISTIC" if spread_cy == 0 else f"jitter = {spread_cy*LAT_PL_PERIOD_NS:.1f} ns"}')
print()
print('-- FPGA wall time (FPGA + output CDC + DMA + OS poll) ---------------')
print(f'   mean={wall_arr.mean():.1f} us  std={wall_arr.std():.1f} us  p2p={wall_arr.max()-wall_arr.min():.1f} us')
print()
print('-- IOC overhead (OS scheduler + DMA handshake + poll granularity) ---')
print(f'   mean={oh_arr.mean():.1f} us  std={oh_arr.std():.1f} us  p2p={oh_arr.max()-oh_arr.min():.1f} us')
print()
print('-- Cache flush (sync_to_device) -------------------------------------')
print(f'   mean={cf_arr.mean():.1f} us  std={cf_arr.std():.1f} us  p2p={cf_arr.max()-cf_arr.min():.1f} us')
print()
print('-- Total end-to-end -------------------------------------------------')
print(f'   mean={tot_arr.mean():.3f} ms  std={tot_arr.std():.3f} ms  p2p={tot_arr.max()-tot_arr.min():.3f} ms')

# Plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle(f'Determinism analysis -- {TEST_FRAME}  (N={N_RUNS})', fontsize=12)

# A: FPGA PL cycles -- should be a single spike
ax = axes[0]
uniq, cnts = np.unique(pl_arr, return_counts=True)
bar_w = max(1, (int(uniq.max()) - int(uniq.min())) // 2) if len(uniq) > 1 else 1
ax.bar(uniq, cnts, color='#1D9E75', width=bar_w)
ax.set_xlabel('delta_pipe (cycles @ 520 MHz)')
ax.set_ylabel('count')
ax.set_title(f'FPGA PL pipeline\nspread = {spread_cy} cycles = {spread_cy*LAT_PL_PERIOD_NS:.1f} ns')

# B: IOC overhead -- OS jitter lives here
ax = axes[1]
ax.hist(oh_arr, bins=min(20, N_RUNS), color='#E24B4A', edgecolor='none', alpha=0.85)
ax.axvline(oh_arr.mean(), color='white', lw=1.5, ls='--',
           label=f'mean={oh_arr.mean():.0f} us')
ax.set_xlabel('IOC overhead (us)  -- OS + DMA + poll jitter')
ax.set_ylabel('count')
ax.set_title('SW overhead jitter\n(irreducible OS noise)')
ax.legend(fontsize=8)

# C: Stacked latency budget (mean of N runs)
ax = axes[2]
last_sw = sw_log.get(TEST_FRAME)
categories = ['Cache flush', 'DMA arm', 'VDMA start', 'FPGA PL (det.)',
              'IOC overhead', 'VDMA stop', 'Cache inv', 'Decode']
means_us = [
    cf_arr.mean(),
    (last_sw.dma_arm_ns   if last_sw else 0) / 1000,
    (last_sw.vdma_start_ns if last_sw else 0) / 1000,
    pl_arr.mean() * LAT_PL_PERIOD_NS / 1000,
    oh_arr.mean(),
    (last_sw.vdma_stop_ns  if last_sw else 0) / 1000,
    (last_sw.cache_inv_ns  if last_sw else 0) / 1000,
    (last_sw.decode_ns     if last_sw else 0) / 1000,
]
colors = ['#378ADD','#185FA5','#0C447C','#1D9E75','#E24B4A','#EF9F27','#BA7517','#888780']
bottom = 0
for lbl, val, col in zip(categories, means_us, colors):
    ax.bar(0, val, bottom=bottom, color=col, width=0.5, label=f'{lbl} ({val:.0f} us)')
    if val > 5:
        ax.text(0, bottom + val/2, f'{val:.0f}', ha='center', va='center',
                color='white', fontsize=7)
    bottom += val
ax.set_ylabel('us')
ax.set_title('Latency budget per frame\n(mean values)')
ax.set_xticks([])
ax.legend(fontsize=6, loc='upper right')

plt.tight_layout()
plt.savefig('latency_determinism.png', dpi=130, bbox_inches='tight')
plt.show()

print()
print('Key insight for deterministic computing:')
print(f'  FPGA PL (delta_pipe) spread : {spread_cy*LAT_PL_PERIOD_NS:.1f} ns  <- hardware deterministic')
print(f'  IOC overhead std            : {oh_arr.std()*1000:.0f} ns  <- OS non-deterministic')
print(f'  Cache flush std             : {cf_arr.std()*1000:.0f} ns  <- CPU non-deterministic')
print()
print('To reduce SW jitter further:')
print('  1. Use UIO interrupt-driven DMA instead of polling (eliminates poll granularity)')
print('  2. Use a PREEMPT_RT kernel (reduces OS scheduling jitter from ~100us to <10us)')
print('  3. Pre-flush cache asynchronously before run_frame() in a background thread')
print('  4. Lock the PYNQ notebook process to an isolated CPU core (taskset / cgroups)')


#### Step 15 - Optional self-test (dark-mode / subtraction / threshold)
Optional hardware self-test of the dark-mode path. **These cells re-capture the
`base_noise` banks** for testing, so if you run them, re-run the SETUP PHASE
(Step 12) before going back to the experiment. Safe to skip in production -
nothing above depends on them. The last cell restores the production baseline.

#### Step 15a - Dark-mode control + capture verification
Checks the control path (write/readback of both registers), then runs a dark
capture with the all-zero frame and with the realistic noisy pedestal frame,
confirming each behaves as a dark capture (no output, ping-pong swap
suppressed), and that active-mode output resumes afterwards.

In [ ]:
print('=== DARK-MODE CONTROL + CAPTURE VERIFICATION ===\n')

# 1. Control-register write / readback --------------------------------------
print('[1] Control register write + readback')
write_dark_mode(DARK_MODE_CAPTURE);  assert read_dark_mode() == 1
write_dark_mode(DARK_MODE_ACTIVE);   assert read_dark_mode() == 0
write_gauss_threshold(1234);         assert read_gauss_threshold() == 1234
write_gauss_threshold(GAUSS_THRESHOLD_DEFAULT)
print('    control path OK\n')

# 2. Dark capture -- all-zero frame -----------------------------------------
print('[2] Dark capture: all-zero frame (baseline reset, no-op subtraction)')
write_dark_mode(DARK_MODE_CAPTURE)
ok_zero, _, _ = run_dark_capture('dark_zero')
write_dark_mode(DARK_MODE_ACTIVE)
assert ok_zero, 'dark_zero capture did not behave as a dark capture'
print()

# 3. Dark capture -- realistic noisy pedestal frame -------------------------
print('[3] Dark capture: realistic noisy dark frame (pedestal + shot noise)')
write_dark_mode(DARK_MODE_CAPTURE)
ok_ped, _, _ = run_dark_capture('dark_pedestal')
write_dark_mode(DARK_MODE_ACTIVE)
assert ok_ped, 'dark_pedestal capture did not behave as a dark capture'
print()

# 4. Active mode restored ----------------------------------------------------
print('[4] Active-mode sanity: pipeline produces output again')
reset_noise_baseline()                       # baseline back to zero for a clean frame
st, sw, hw = run_frame('all_ground', verbose=True)
assert int(np.sum(st)) == 0, 'all_ground should give 0 Rydberg in active mode'
print('    active-mode output restored OK\n')

print('=== DARK-MODE CONTROL + CAPTURE: ALL CHECKS PASSED ===')

#### Step 15b - Noise-subtraction verification
`verify_active` is a deterministic near-threshold frame. With a zero baseline
the near-threshold excited qubits read Rydberg; after a realistic pedestal
baseline is captured, `read_streamer`'s per-pixel saturating subtraction removes
the margin and the same qubits collapse to Ground - while a genuinely bright
frame still reads Rydberg. Decisions are the only hardware-visible output (the
engine `o_score` port is unconnected at the top level), so the frames are
constructed to make the effect crisp through decision bits alone.

In [ ]:
print('=== NOISE-SUBTRACTION VERIFICATION ===\n')
print('verify_active: checkerboard excited set, deterministic near-threshold frame')
print(f'  baseline = 0        -> excited ROI {VERIFY_HI} -> score {16*VERIFY_HI} > {GAUSS_THRESHOLD_DEFAULT} -> Rydberg')
print(f'  baseline = pedestal -> excited ROI {VERIFY_HI}-{DARK_PEDESTAL_DN} ~ {VERIFY_HI-DARK_PEDESTAL_DN}'
      f' -> score ~{16*(VERIFY_HI-DARK_PEDESTAL_DN)} < {GAUSS_THRESHOLD_DEFAULT} -> Ground\n')

write_gauss_threshold(GAUSS_THRESHOLD_DEFAULT)
n_expected = sum(VERIFY_PATTERN)

# Phase A: zero baseline -> subtraction is a no-op
print('[A] baseline = dark_zero  (subtraction = no-op)')
capture_baseline('dark_zero')
states_A, _, _ = run_frame('verify_active', verbose=True)
count_A = int(np.sum(states_A))
print(f'    Rydberg detected = {count_A}/100   (expect {n_expected})\n')

# Phase B: realistic pedestal baseline -> subtraction removes the margin
print('[B] baseline = dark_pedestal  (real per-pixel subtraction)')
capture_baseline('dark_pedestal')
states_B, _, _ = run_frame('verify_active', verbose=True)
count_B = int(np.sum(states_B))
print(f'    Rydberg detected = {count_B}/100   (expect 0)\n')

# Phase B control: a genuinely bright frame must still survive subtraction
print('[B-ctrl] baseline = dark_pedestal, bright frame all_rydberg')
states_Bc, _, _ = run_frame('all_rydberg', verbose=True)
count_Bc = int(np.sum(states_Bc))
print(f'    Rydberg detected = {count_Bc}/100   (expect 100 -- real signal survives)\n')

# restore a clean state for later cells
reset_noise_baseline()

print('--- result ---')
print(f'  baseline 0        : {count_A:3d}/100 Rydberg   (near-threshold qubits detected)')
print(f'  baseline pedestal : {count_B:3d}/100 Rydberg   (near-threshold qubits collapsed)')
print(f'  baseline pedestal : {count_Bc:3d}/100 Rydberg   (bright frame, signal preserved)')
ok = (count_A == n_expected) and (count_B == 0) and (count_Bc == 100)
assert ok, 'noise subtraction did not change results as expected'
print('\n=== NOISE-SUBTRACTION VERIFICATION: PASSED ===')
print('  Subtracting the captured pedestal flipped the near-threshold qubits to')
print('  Ground while leaving the bright frame untouched -- read_streamer per-pixel')
print('  saturating subtraction (out = max(0, roi - noise)) is active and correct.')

#### Step 15c - Runtime threshold verification
With a zero baseline and the deterministic `verify_active` frame the excited
and ground ROI scores are known exactly (`16 * VERIFY_HI` and
`16 * DARK_PEDESTAL_DN`). Sweeping `gaussian_threshold` below / between / above
those two scores drives the Rydberg count 100 -> 50 -> 0, proving the runtime
threshold register gates the decision.

In [ ]:
print('=== RUNTIME THRESHOLD VERIFICATION ===\n')

reset_noise_baseline()                       # baseline = 0 -> scores are exact
n_expected = sum(VERIFY_PATTERN)
score_exc  = 16 * VERIFY_HI                  # excited-qubit ROI score
score_gnd  = 16 * DARK_PEDESTAL_DN           # ground-qubit ROI score
print(f'verify_active with baseline = 0:  excited score = {score_exc}, ground score = {score_gnd}\n')

# threshold below both scores -> everything Rydberg
# threshold between the two   -> only excited  -> n_expected
# threshold above both scores -> everything Ground
THRESH_LOW  = score_gnd - 50
THRESH_MID  = (score_gnd + score_exc) // 2
THRESH_HIGH = score_exc + 50

results = {}
for label, th, exp in [('LOW',  THRESH_LOW,  100),
                       ('MID',  THRESH_MID,  n_expected),
                       ('HIGH', THRESH_HIGH, 0)]:
    write_gauss_threshold(th, verify=True)
    st, _, _ = run_frame('verify_active', verbose=False)
    cnt = int(np.sum(st))
    results[label] = cnt
    flag = 'OK' if cnt == exp else '*** FAIL ***'
    print(f'  threshold={th:5d} ({label:<4})  ->  {cnt:3d}/100 Rydberg   (expect {exp:3d})  {flag}')

write_gauss_threshold(GAUSS_THRESHOLD_DEFAULT)   # restore default
print(f'\n  threshold restored to default ({GAUSS_THRESHOLD_DEFAULT})')

ok = (results['LOW'] == 100 and results['MID'] == n_expected and results['HIGH'] == 0)
assert ok, 'threshold did not gate decisions as expected'
print(f'\n=== RUNTIME THRESHOLD VERIFICATION: PASSED ===')
print(f'  Moving gaussian_threshold across the known excited/ground scores drives')
print(f'  the Rydberg count 100 -> {n_expected} -> 0 as expected.')

# --- leave the rig in the production state: re-capture the base-noise baseline
print()
print('Restoring production state: re-running the SETUP PHASE base-noise capture ...')
capture_baseline(globals().get('BASE_NOISE_FRAME', 'dark_pedestal'))
print(f'Production baseline restored.  dark_mode = {read_dark_mode()}')

#### Step 16 - Cleanup


In [ ]:
vdma_stop()
frame_buf.freebuffer()
result_buf.freebuffer()
if 'PINNED' in globals():
    for _name, _buf in PINNED.items():
        _buf.freebuffer()
    PINNED.clear()
    print(f'Released {len(_HOT_FRAMES)} pinned CMA buffers.')
print('VDMA stopped. Buffers released.')
